# Integrating Qwen 2.5 into AutoLLMTuner

This notebook demonstrates how to integrate the Qwen/Qwen2.5-7B-Instruct model into the AutoLLMTuner framework for quantization optimization.

## Section 1: Import Required Libraries

Import necessary libraries from transformers, torch, and the AutoLLMTuner framework.

In [ ]:
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Add src to path for importing AutoLLMTuner modules
sys.path.insert(0, '/Users/jonathan/Documents/Desktop/GT-Master-Files/Fall_2025/Big_Data/AutoLLMTuner/src')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Section 2: Load Qwen 2.5 Model and Tokenizer

Load the Qwen/Qwen2.5-7B-Instruct model and inspect its architecture.

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading tokenizer and model from: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Set pad token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model loaded successfully")
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

In [ ]:
# Display model architecture
print("=" * 60)
print("Model Architecture")
print("=" * 60)
print(f"Number of layers: {len(model.model.layers)}")
print("\nModel structure:")
print(model)

In [ ]:
# Calculate model size and parameters
total_params = sum(p.numel() for p in model.parameters())
total_size_mb = sum(p.nelement() * p.element_size() for p in model.parameters()) / 1024**2

print("=" * 60)
print("Model Size & Parameters")
print("=" * 60)
print(f"Total parameters: {total_params:,}")
print(f"Total size (MB): {total_size_mb:,.2f}")
print(f"Total size (GB): {total_size_mb / 1024:,.2f}")

## Section 3: Create Qwen Inference Wrapper Class

Define a wrapper class to handle Qwen model inference with chat templates.

In [ ]:
class QwenInference:
    """Wrapper for Qwen2.5 inference with chat template support"""
    
    def __init__(self, model, tokenizer):
        """Initialize with model and tokenizer instances"""
        self.model = model
        self.tokenizer = tokenizer
        
    def format_prompt(self, system_prompt: str, user_message: str) -> str:
        """Format message using Qwen's chat template"""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    
    def generate(self, 
                 system_prompt: str, 
                 user_message: str, 
                 max_new_tokens: int = 512,
                 temperature: float = 0.7,
                 top_p: float = 0.9) -> str:
        """Generate response to a prompt"""
        
        # Format the prompt
        text = self.format_prompt(system_prompt, user_message)
        
        # Tokenize
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True
            )
        
        # Decode
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response
    
    def count_tokens(self, text: str) -> int:
        """Count tokens in text"""
        return len(self.tokenizer.encode(text))

# Create inference wrapper
qwen_inf = QwenInference(model, tokenizer)
print("✓ QwenInference wrapper created successfully")

## Section 4: Test Model Integration

Test the model with sample prompts using chat templates (mimicking your notebook example).

In [ ]:
# Test 1: Simple math question
system_prompt = "You are a math teacher who helps students with their math questions."

user_message = r"""Define
\[p = \sum_{k = 1}^\infty \frac{1}{k^2} \quad \text{and} \quad q = \sum_{k = 1}^\infty \frac{1}{k^3}.\]Find a way to write
\[\sum_{j = 1}^\infty \sum_{k = 1}^\infty \frac{1}{(j + k)^3}\]in terms of $p$ and $q.$
"""

print("=" * 60)
print("Test 1: Math Question")
print("=" * 60)
print("System Prompt:", system_prompt)
print("\nUser Question:", user_message[:100] + "...")
print("\nGenerating response...")

response = qwen_inf.generate(system_prompt, user_message, max_new_tokens=512)
print("\nResponse:")
print(response)

In [ ]:
# Test 2: Simple test question
test_question = "What is 2+2?"
response2 = qwen_inf.generate("You are a helpful assistant.", test_question, max_new_tokens=128)

print("\n" + "=" * 60)
print("Test 2: Simple Question")
print("=" * 60)
print(f"Question: {test_question}")
print(f"Response: {response2}")

# Test 3: Token counting
token_count = qwen_inf.count_tokens(user_message)
print("\n" + "=" * 60)
print("Test 3: Token Counting")
print("=" * 60)
print(f"Tokens in original question: {token_count}")

## Section 5: Using Qwen with AutoLLMTuner Framework

Integrate Qwen with the existing `HFModelLoader` in your AutoLLMTuner project.

In [ ]:
from models.llama_loader import HFModelLoader

# Example: Using HFModelLoader with Qwen config
# This is how you would use it in main_search.py

config_path = "configs/qwen2.5_run_config.yaml"
layer_spec_path = "configs/qwen2.5_layer_spec.yaml"

# Instantiate loader
loader = HFModelLoader(config_path, layer_spec_path)

# Load model and tokenizer
model_loaded, tokenizer_loaded = loader.load_model()

print("=" * 60)
print("Integration with HFModelLoader")
print("=" * 60)
print(f"✓ Model loaded from HFModelLoader")
print(f"Model type: {type(model_loaded)}")
print(f"Tokenizer type: {type(tokenizer_loaded)}")

## Section 6: Benchmark Model Parameters and Memory

Calculate and compare model parameters and memory footprint for Qwen

In [ ]:
import pandas as pd

# Get layer info for Qwen
layer_info = {}
for i, layer in enumerate(model.model.layers):
    layer_params = sum(p.numel() for p in layer.parameters())
    layer_info[f"layer_{i}"] = layer_params

# Summary statistics
num_layers = len(model.model.layers)
params_per_layer = [layer_info[f"layer_{i}"] for i in range(num_layers)]

print("=" * 60)
print("Qwen2.5-7B Model Benchmarks")
print("=" * 60)
print(f"Model: {model_name}")
print(f"Total Parameters: {total_params:,}")
print(f"Total Size (float16): {total_size_mb:,.2f} MB ({total_size_mb / 1024:,.2f} GB)")
print(f"Number of Layers: {num_layers}")
print(f"Avg Parameters per Layer: {sum(params_per_layer) // num_layers:,}")
print(f"Hidden Dimension: {model.model.hidden_size}")
print(f"Intermediate Size: {model.model.intermediate_size if hasattr(model.model, 'intermediate_size') else 'N/A'}")
print(f"Number of Attention Heads: {model.model.num_attention_heads}")
print(f"Vocab Size: {model.model.vocab_size}")

In [ ]:
# Layer-wise breakdown
print("\n" + "=" * 60)
print("Layer-wise Parameter Distribution")
print("=" * 60)
layer_df = pd.DataFrame({
    "Layer": list(range(num_layers)),
    "Parameters": params_per_layer,
    "Size_MB": [p * 2 / 1024**2 for p in params_per_layer]
})

print(layer_df.to_string())
print(f"\nMean params per layer: {layer_df['Parameters'].mean():,.0f}")
print(f"Std dev: {layer_df['Parameters'].std():,.0f}")